# DistilBERT Intent Classifier

This notebook trains and evaluates a DistilBERT-based intent classifier
for the ResolveAI customer-support dataset.

The model is trained using the corrected leakage-safe generic-placeholder
training split and evaluated on the corresponding validation split.

The official test split remains untouched during model development.

After model selection, the frozen model will also be evaluated on the
independent OOD challenge benchmark to measure generalization to more
realistic customer language.

In [1]:
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
)

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [3]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Using device:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

Using device: cpu


### Model Configuration

In [4]:
MODEL_NAME = "distilbert-base-uncased"

TEXT_COLUMN = "processed_instruction"
TARGET_COLUMN = "intent"

### Load train and validation

In [5]:
train_df = pd.read_csv(
    "../data/processed/generic/train.csv"
)

validation_df = pd.read_csv(
    "../data/processed/generic/validation.csv"
)

print(
    "Train shape:",
    train_df.shape
)

print(
    "Validation shape:",
    validation_df.shape
)

Train shape: (18780, 9)
Validation shape: (4016, 9)


#### Verifying required columns

In [6]:
print(
    train_df.columns.tolist()
)

['flags', 'instruction', 'category', 'intent', 'response', 'normalized_instruction', 'split_group_key', 'processed_instruction', 'split']


In [7]:
required_columns = {
    TEXT_COLUMN,
    TARGET_COLUMN,
}

assert required_columns.issubset(
    train_df.columns
)

assert required_columns.issubset(
    validation_df.columns
)

print(
    "Required columns found."
)

Required columns found.


In [8]:
print(
    train_df[
        [
            TEXT_COLUMN,
            TARGET_COLUMN
        ]
    ]
    .isna()
    .sum()
)

print()

print(
    validation_df[
        [
            TEXT_COLUMN,
            TARGET_COLUMN
        ]
    ]
    .isna()
    .sum()
)

processed_instruction    0
intent                   0
dtype: int64

processed_instruction    0
intent                   0
dtype: int64


#### Check intents

In [9]:
print(
    "Train intents:",
    train_df[TARGET_COLUMN].nunique()
)

print(
    "Validation intents:",
    validation_df[TARGET_COLUMN].nunique()
)

Train intents: 27
Validation intents: 27


In [10]:
sorted(
    train_df[TARGET_COLUMN].unique()
)

['cancel_order',
 'change_order',
 'change_shipping_address',
 'check_cancellation_fee',
 'check_invoice',
 'check_payment_methods',
 'check_refund_policy',
 'complaint',
 'contact_customer_service',
 'contact_human_agent',
 'create_account',
 'delete_account',
 'delivery_options',
 'delivery_period',
 'edit_account',
 'get_invoice',
 'get_refund',
 'newsletter_subscription',
 'payment_issue',
 'place_order',
 'recover_password',
 'registration_problems',
 'review',
 'set_up_shipping_address',
 'switch_account',
 'track_order',
 'track_refund']

### Create ```label2id``` and ```id2label```

In [11]:
# Creating deterministic mapping

intent_labels = sorted(
  train_df[TARGET_COLUMN].unique()
)

label2id = {
  label: idx
  for idx, label in enumerate(
    intent_labels
  )
}

id2label = {
  idx: label
  for label, idx in label2id.items()
}

print(
  "Number of labels: ",
  len(label2id)
)

Number of labels:  27


In [12]:
label2id

{'cancel_order': 0,
 'change_order': 1,
 'change_shipping_address': 2,
 'check_cancellation_fee': 3,
 'check_invoice': 4,
 'check_payment_methods': 5,
 'check_refund_policy': 6,
 'complaint': 7,
 'contact_customer_service': 8,
 'contact_human_agent': 9,
 'create_account': 10,
 'delete_account': 11,
 'delivery_options': 12,
 'delivery_period': 13,
 'edit_account': 14,
 'get_invoice': 15,
 'get_refund': 16,
 'newsletter_subscription': 17,
 'payment_issue': 18,
 'place_order': 19,
 'recover_password': 20,
 'registration_problems': 21,
 'review': 22,
 'set_up_shipping_address': 23,
 'switch_account': 24,
 'track_order': 25,
 'track_refund': 26}

In [13]:
train_df["label"] = (
    train_df[TARGET_COLUMN]
    .map(label2id)
)

validation_df["label"] = (
    validation_df[TARGET_COLUMN]
    .map(label2id)
)

In [14]:
train_df[
    [
        TEXT_COLUMN,
        TARGET_COLUMN,
        "label"
    ]
].head()

,processed_instruction,intent,label
0,question about cancelling order ENTITY_TOKEN,cancel_order,0
1,i need help cancelling puchase ENTITY_TOKEN,cancel_order,0
2,I need to cancel purchase ENTITY_TOKEN,cancel_order,0
3,can you help me cancel order ENTITY_TOKEN?,cancel_order,0
4,"I can no longer afford order ENTITY_TOKEN, can...",cancel_order,0


In [15]:
assert train_df["label"].notna().all()
assert validation_df["label"].notna().all()

assert train_df["label"].nunique() == 27
assert validation_df["label"].nunique() == 27

print(
    "Label encoding validated."
)

Label encoding validated.


## Convert Pandas DataFrames to Hugging Face Datasets

In [16]:
train_dataset = Dataset.from_pandas(
    train_df[
        [
            TEXT_COLUMN,
            "label"
        ]
    ],
    preserve_index=False
)

validation_dataset = Dataset.from_pandas(
    validation_df[
        [
            TEXT_COLUMN,
            "label"
        ]
    ],
    preserve_index=False
)

In [17]:
print(train_dataset)
print(validation_dataset)

Dataset({
    features: ['processed_instruction', 'label'],
    num_rows: 18780
})
Dataset({
    features: ['processed_instruction', 'label'],
    num_rows: 4016
})


In [18]:
train_dataset[0]

{'processed_instruction': 'question about cancelling order ENTITY_TOKEN',
 'label': 0}